In [12]:
# Cell 1: imports and load
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from src.cleaning_utils import normalize_id_columns
from src.paths import RAW_DIR, ensure_parent

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)

raw_path = RAW_DIR / "v_add_student_degree_status.parquet"


In [13]:
df=pd.read_parquet(raw_path)

In [14]:
df['permanent_status_id'].value_counts()

permanent_status_id
2.0     153972
22.0     10503
10.0      5938
31.0      5019
17.0      2434
41.0      1873
24.0      1688
9.0       1146
15.0       621
18.0       607
27.0       347
12.0       147
40.0       100
16.0        75
28.0        50
13.0         7
39.0         1
19.0         1
1.0          1
Name: count, dtype: int64

In [15]:
drop_permenant_status_variables=[1,4,11,12,15,16,41]

In [16]:
d1=df.drop(df[df['permanent_status_id'].isin(drop_permenant_status_variables)].index,)

In [17]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 184530 entries, 0 to 184529
Data columns (total 44 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   student_status_id       184530 non-null  float64
 1   student_id              184530 non-null  float64
 2   part_id                 184530 non-null  float64
 3   degree_id               182404 non-null  float64
 4   start_part_id           184525 non-null  float64
 5   finish_part_id          117154 non-null  float64
 6   grade_version_id        184519 non-null  float64
 7   permanent_status_id     184530 non-null  float64
 8   permanent_status_sl     184530 non-null  str    
 9   study_mode              184530 non-null  str    
 10  prev_gpa_points         164485 non-null  float64
 11  prev_gpa_percent        164485 non-null  float64
 12  gpa_percent             184519 non-null  float64
 13  gpa_points              184519 non-null  float64
 14  start_agpa_percent      184519 

In [18]:
d1.info()

<class 'pandas.DataFrame'>
Index: 181813 entries, 0 to 184529
Data columns (total 44 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   student_status_id       181813 non-null  float64
 1   student_id              181813 non-null  float64
 2   part_id                 181813 non-null  float64
 3   degree_id               181720 non-null  float64
 4   start_part_id           181813 non-null  float64
 5   finish_part_id          117149 non-null  float64
 6   grade_version_id        181813 non-null  float64
 7   permanent_status_id     181813 non-null  float64
 8   permanent_status_sl     181813 non-null  str    
 9   study_mode              181813 non-null  str    
 10  prev_gpa_points         164483 non-null  float64
 11  prev_gpa_percent        164483 non-null  float64
 12  gpa_percent             181813 non-null  float64
 13  gpa_points              181813 non-null  float64
 14  start_agpa_percent      181813 non-n

In [19]:
a=df[df['permanent_status_id'].isin(drop_permenant_status_variables)]
a['permanent_status_sl'].value_counts()

permanent_status_sl
مستنكف - مفاضلة              1873
متقدم - مفاضلة                621
قيد التسجيل - مفاضلة          147
قيد التسجيل - تسجيل مباشر      75
متقدم - تسجيل مباشر             1
Name: count, dtype: int64

In [20]:
import pandas as pd

# =========================
# 1) ضع أسماء الجداول هنا
clean_student_course_attempts=pd.read_parquet(r"D:\AI\Real projects\Academic_Advisor\data\preprocessed\V_CRG_STUDENT_COURSE\clean_v_crg_student_course.parquet")
clean_student_degree_status=pd.read_parquet(RAW_DIR / "v_add_student_degree_status.parquet")
# =========================

df_crg = clean_student_course_attempts.copy()
df_add =d1.copy()  # استخدم d1 بعد التنظيف

# =========================
# 2) دالة تنظيف student_id
# =========================

def normalize_id_for_compare(series: pd.Series) -> pd.Series:
    """
    يحول student_id إلى string ويحافظ على suffix مثل .111
    """
    return (
        series
        .astype("string")
        .str.strip()
        .replace("", pd.NA)
    )

# =========================
# 3) تجهيز student_id في الجدولين
# =========================

crg_students = (
    normalize_id_for_compare(df_crg["student_id"])
    .dropna()
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

add_students = (
    normalize_id_for_compare(df_add["student_id"])
    .dropna()
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

# =========================
# 4) تحويل إلى sets للمقارنة
# =========================

crg_set = set(crg_students)
add_set = set(add_students)

students_in_both = sorted(crg_set & add_set)
students_only_in_crg = sorted(crg_set - add_set)
students_only_in_add = sorted(add_set - crg_set)

# =========================
# 5) تحويل النتائج إلى DataFrames
# =========================

df_students_in_both = pd.DataFrame({"student_id": students_in_both})
df_students_only_in_crg = pd.DataFrame({"student_id": students_only_in_crg})
df_students_only_in_add = pd.DataFrame({"student_id": students_only_in_add})

# =========================
# 6) Summary
# =========================

student_overlap_summary = pd.DataFrame([{
    "crg_unique_students": len(crg_set),
    "add_unique_students": len(add_set),
    "students_in_both": len(students_in_both),
    "students_only_in_crg": len(students_only_in_crg),
    "students_only_in_add": len(students_only_in_add),
    "crg_match_rate_vs_crg": round(len(students_in_both) / len(crg_set) * 100, 2) if crg_set else 0,
    "add_match_rate_vs_add": round(len(students_in_both) / len(add_set) * 100, 2) if add_set else 0,
}])

display(student_overlap_summary)

print("Students in both:")
display(df_students_in_both.head(50))

print("Students only in CRG Student Course:")
display(df_students_only_in_crg.head(50))

print("Students only in ADD Student Degree Status:")
display(df_students_only_in_add.head(50))

,crg_unique_students,add_unique_students,students_in_both,students_only_in_crg,students_only_in_add,crg_match_rate_vs_crg,add_match_rate_vs_add
0,16171,17416,16171,0,1245,100.0,92.85


Students in both:


,student_id
0,10000.111
1,10001.111
2,10002.111
3,10003.111
4,10005.111
5,10007.111
6,10008.111
7,10009.111
8,10011.111
9,10012.111


Students only in CRG Student Course:


,student_id


Students only in ADD Student Degree Status:


,student_id
0,10004.111
1,10006.111
2,10016.111
3,10044.111
4,10064.111
5,10065.111
6,10081.111
7,10085.111
8,10098.111
9,10127.111


In [36]:
d1[d1['finish_status'].eq('CHANGE_DEGREE')]

,student_status_id,student_id,part_id,degree_id,start_part_id,finish_part_id,grade_version_id,permanent_status_id,permanent_status_sl,study_mode,prev_gpa_points,prev_gpa_percent,gpa_percent,gpa_points,start_agpa_percent,start_agpa_points,start_total_in_courses,start_total_in_credits,end_total_in_courses,end_total_in_credits,end_agpa_percent,end_agpa_points,semester_reg_courses,semester_reg_credits,semester_pass_courses,semester_pass_credits,semester_fail_courses,semester_fail_credits,semester_in_courses,semester_in_credits,total_semesters,total_reg_courses,total_reg_credits,total_pass_courses,total_pass_credits,total_fail_courses,total_fail_credits,reg_total_semesters,finish_status,version_title_sl,degree_name_sl,degree_credits_count,start_level_id,start_level_name_pl
534,156016.111,167.111,20111.0,19.111,20111.0,20134.0,3.111,2.0,مقبول,C,NaN,NaN,66.2,2.31,0.0,0.00,0.0,0.0,7.0,17.0,66.2,2.31,7.0,17.0,7.0,17.0,0.0,0.0,7.0,17.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,CHANGE_DEGREE,القرار رقم 230,إدارة الأعمال,148.0,236.111,First Year
535,156017.111,167.111,20112.0,19.111,20111.0,20134.0,3.111,2.0,مقبول,C,2.31,66.2,61.4,2.07,66.2,2.31,7.0,17.0,15.0,35.0,63.8,2.19,8.0,18.0,8.0,18.0,0.0,0.0,8.0,18.0,2.0,7.0,17.0,7.0,17.0,0.0,0.0,2.0,CHANGE_DEGREE,القرار رقم 230,إدارة الأعمال,148.0,236.111,First Year
536,213707.111,167.111,20121.0,19.111,20111.0,20134.0,3.111,2.0,مقبول,C,2.07,61.4,0.0,0.00,63.8,2.19,15.0,35.0,15.0,35.0,63.8,2.19,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,15.0,35.0,15.0,35.0,0.0,0.0,2.0,CHANGE_DEGREE,القرار رقم 230,إدارة الأعمال,148.0,237.111,Second Year
537,144891.111,167.111,20122.0,19.111,20111.0,20134.0,3.111,2.0,مقبول,C,0.00,0.0,0.0,0.00,63.8,2.19,15.0,35.0,15.0,35.0,63.8,2.19,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,15.0,35.0,15.0,35.0,0.0,0.0,2.0,CHANGE_DEGREE,القرار رقم 230,إدارة الأعمال,148.0,237.111,Second Year
538,156018.111,167.111,20131.0,19.111,20111.0,20134.0,3.111,2.0,مقبول,C,0.00,0.0,70.6,2.53,63.8,2.19,15.0,35.0,22.0,53.0,66.0,2.30,7.0,18.0,7.0,18.0,0.0,0.0,7.0,18.0,5.0,15.0,35.0,15.0,35.0,0.0,0.0,3.0,CHANGE_DEGREE,القرار رقم 230,إدارة الأعمال,148.0,237.111,Second Year
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
174101,365005.111,30135.111,20232.0,49.111,20231.0,20234.0,3.111,22.0,مفصول من الاختصاص - أكاديمياً,C,0.00,0.0,0.0,0.00,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.00,4.0,12.0,0.0,0.0,4.0,12.0,0.0,0.0,2.0,6.0,18.0,0.0,0.0,6.0,18.0,2.0,CHANGE_DEGREE,القرار رقم 230,هندسة الاتصالات 2023,173.0,2056.111,First Year
174414,357930.111,30189.111,20231.0,48.111,20231.0,20232.0,3.111,2.0,مقبول,C,NaN,NaN,0.0,0.00,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.00,5.0,15.0,0.0,0.0,5.0,15.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,CHANGE_DEGREE,القرار رقم 230,الهندسة المعلوماتية/هندسة أمن النظم والشبكات ا...,173.0,2044.111,First Year
174415,364944.111,30189.111,20232.0,48.111,20231.0,20232.0,3.111,22.0,مفصول من الاختصاص - أكاديمياً,C,0.00,0.0,36.2,0.81,0.0,0.00,0.0,0.0,2.0,6.0,33.0,0.65,4.0,12.0,2.0,6.0,2.0,6.0,2.0,6.0,2.0,5.0,15.0,0.0,0.0,5.0,15.0,2.0,CHANGE_DEGREE,القرار رقم 230,الهندسة المعلوماتية/هندسة أمن النظم والشبكات ا...,173.0,2044.111,First Year
174522,357952.111,30211.111,20231.0,48.111,20231.0,20232.0,3.111,2.0,مقبول,C,NaN,NaN,0.0,0.00,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.00,6.0,18.0,0.0,0.0,6.0,18.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,CHANGE_DEGREE,القرار رقم 230,الهندسة المعلوماتية/هندسة أمن النظم والشبكات ا...,173.0,2044.111,First Year


In [25]:
d1['finish_status'].value_counts(dropna=False) 

finish_status
GRADUATED                  95220
NaN                        64348
CLOSE_FILE                 14395
WITHDRAWN                   4200
CHANGE_PLAN                 1885
CANCEL_ADMISSION             604
FINAL_DISMISS                539
CHANGE_DEGREE                380
CHANGE_FACULTY               238
COMPARISON_UNDERAPPLIED        4
Name: count, dtype: int64

In [26]:
d1['permanent_status_sl'].value_counts()

permanent_status_sl
مقبول                            153972
مفصول من الاختصاص - أكاديمياً     10503
متخرج                              5938
مفصول - استدراكي                   5019
ترقين قيد                          2434
مفصول من الجامعة - أكاديمياً       1688
منسحب                              1146
إلغاء الانتساب                      607
الامتحان الوطني                     347
راسب مفاضلة                         100
فصل نهائي                            50
تغيير قيد                             7
مفصول من الاختصاص                     1
مستنفذ                                1
Name: count, dtype: int64

In [28]:
d1[d1['permanent_status_sl']=='الامتحان الوطني']

,student_status_id,student_id,part_id,degree_id,start_part_id,finish_part_id,grade_version_id,permanent_status_id,permanent_status_sl,study_mode,prev_gpa_points,prev_gpa_percent,gpa_percent,gpa_points,start_agpa_percent,start_agpa_points,start_total_in_courses,start_total_in_credits,end_total_in_courses,end_total_in_credits,end_agpa_percent,end_agpa_points,semester_reg_courses,semester_reg_credits,semester_pass_courses,semester_pass_credits,semester_fail_courses,semester_fail_credits,semester_in_courses,semester_in_credits,total_semesters,total_reg_courses,total_reg_credits,total_pass_courses,total_pass_credits,total_fail_courses,total_fail_credits,reg_total_semesters,finish_status,version_title_sl,degree_name_sl,degree_credits_count,start_level_id,start_level_name_pl
8786,291832.111,3309.111,20201.0,2.111,20131.0,20202.0,3.111,27.0,الامتحان الوطني,C,2.12,62.4,60.0,2.00,52.8,1.64,61.0,224.0,62.0,248.0,58.4,1.92,1.0,24.0,1.0,24.0,0.0,0.0,1.0,24.0,41.0,215.0,813.5,73.0,257.0,131.0,525.0,41.0,GRADUATED,القرار رقم 230,دكتور في الطب,251.0,557.111,Sixth Year
8832,389228.111,3348.111,20242.0,2.111,20131.0,20243.0,3.111,27.0,الامتحان الوطني,C,0.00,0.0,65.0,2.25,63.4,2.17,62.0,227.0,63.0,251.0,63.6,2.18,1.0,24.0,1.0,24.0,0.0,0.0,1.0,24.0,44.0,180.0,646.0,74.0,255.0,91.0,341.0,39.0,GRADUATED,القرار رقم 230,دكتور في الطب,251.0,557.111,Sixth Year
8886,314160.111,3432.111,20211.0,2.111,20131.0,20212.0,3.111,27.0,الامتحان الوطني,C,0.00,0.0,60.0,2.00,60.6,2.03,61.0,224.0,62.0,248.0,60.6,2.03,1.0,24.0,1.0,24.0,0.0,0.0,1.0,24.0,38.0,134.0,474.0,77.0,275.0,46.0,166.0,33.0,GRADUATED,القرار رقم 230,دكتور في الطب,251.0,557.111,Sixth Year
8908,358127.111,3470.111,20231.0,2.111,20161.0,20232.0,3.111,27.0,الامتحان الوطني,C,0.00,0.0,0.0,0.00,62.2,2.11,62.0,227.0,62.0,227.0,62.2,2.11,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,48.0,278.0,996.0,77.0,260.5,193.0,713.5,48.0,GRADUATED,القرار رقم 230,دكتور في الطب,251.0,557.111,Sixth Year
8945,358404.111,3472.111,20232.0,2.111,20131.0,NaN,3.111,27.0,الامتحان الوطني,C,2.24,64.8,25.8,0.29,57.8,1.89,60.0,222.0,62.0,227.0,53.2,1.66,10.0,48.0,2.0,5.0,3.0,31.0,2.0,5.0,49.0,225.0,897.5,68.0,245.5,145.0,606.5,46.0,NaN,القرار رقم 230,دكتور في الطب,251.0,557.111,Sixth Year
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
112361,358257.111,14556.111,20232.0,2.111,20191.0,20233.0,3.111,27.0,الامتحان الوطني,C,3.44,88.8,70.0,2.50,80.6,3.03,61.0,224.0,62.0,248.0,79.6,2.98,7.0,36.0,1.0,24.0,0.0,0.0,1.0,24.0,14.0,61.0,231.0,57.0,214.0,0.0,0.0,14.0,GRADUATED,القرار رقم 230,دكتور في الطب,251.0,587.111,Sixth Year
113873,360848.111,14672.111,20232.0,2.111,20191.0,20233.0,3.111,27.0,الامتحان الوطني,C,3.30,86.0,66.4,2.32,70.4,2.52,59.0,215.0,63.0,251.0,70.4,2.52,7.0,48.0,4.0,36.0,0.0,0.0,4.0,36.0,13.0,65.0,236.0,58.0,218.0,1.0,6.0,13.0,GRADUATED,القرار رقم 230,دكتور في الطب,251.0,587.111,Sixth Year
114439,389295.111,14720.111,20242.0,2.111,20191.0,20251.0,3.111,27.0,الامتحان الوطني,C,0.00,0.0,60.0,2.00,0.0,0.00,61.0,224.0,62.0,248.0,67.4,2.37,1.0,24.0,1.0,24.0,0.0,0.0,1.0,24.0,16.0,75.0,270.5,62.0,232.0,3.0,12.5,16.0,GRADUATED,القرار رقم 230,دكتور في الطب,251.0,587.111,Sixth Year
117551,389302.111,15015.111,20242.0,2.111,20201.0,20251.0,3.111,27.0,الامتحان الوطني,C,2.00,60.0,0.0,0.00,0.0,0.00,61.0,224.0,61.0,224.0,59.4,1.97,1.0,24.0,0.0,0.0,1.0,24.0,0.0,0.0,13.0,57.0,204.0,43.0,167.0,4.0,10.0,13.0,GRADUATED,القرار رقم 230,دكتور في الطب,251.0,587.111,Sixth Year


In [37]:
no_activity = (
    (d1["total_reg_courses"].fillna(0) == 0) &
    (d1["total_reg_credits"].fillna(0) == 0) &
    (d1["total_pass_courses"].fillna(0) == 0) &
    (d1["total_pass_credits"].fillna(0) == 0) &
    (d1["total_fail_courses"].fillna(0) == 0) &
    (d1["total_fail_credits"].fillna(0) == 0)
)

print("Total no activity rows:", no_activity.sum())

print("\nfinish_status distribution for no activity rows:")
print(d1.loc[no_activity, "finish_status"].value_counts(dropna=False))

print("\nfinish_status distribution for with activity rows:")
print(d1.loc[~no_activity, "finish_status"].value_counts(dropna=False))

Total no activity rows: 17749

finish_status distribution for no activity rows:
finish_status
NaN                        7376
GRADUATED                  5689
CLOSE_FILE                 2235
WITHDRAWN                  1074
CANCEL_ADMISSION            604
CHANGE_PLAN                 461
CHANGE_FACULTY              115
CHANGE_DEGREE               113
FINAL_DISMISS                81
COMPARISON_UNDERAPPLIED       1
Name: count, dtype: int64

finish_status distribution for with activity rows:
finish_status
GRADUATED                  89531
NaN                        56972
CLOSE_FILE                 12160
WITHDRAWN                   3126
CHANGE_PLAN                 1424
FINAL_DISMISS                458
CHANGE_DEGREE                267
CHANGE_FACULTY               123
COMPARISON_UNDERAPPLIED        3
Name: count, dtype: int64


In [38]:
student_degree_counts = (
    d1.dropna(subset=["degree_id"])
    .groupby("student_id")["degree_id"]
    .nunique()
    .sort_values(ascending=False)
)

print("Students with more than one degree_id:", (student_degree_counts > 1).sum())
print(student_degree_counts.value_counts().sort_index())

multi_degree_students = student_degree_counts[student_degree_counts > 1].index

display(
    d1[d1["student_id"].isin(multi_degree_students)]
    .sort_values(["student_id", "part_id"])
    [
        [
            "student_id", "part_id", "start_part_id", "finish_part_id",
            "degree_id", "degree_name_sl", "finish_status",
            "gpa_points", "end_agpa_points",
            "total_reg_credits", "total_pass_credits"
        ]
    ]
    .head(150)
)

Students with more than one degree_id: 761
degree_id
1    16562
2      734
3       25
4        2
Name: count, dtype: int64


,student_id,part_id,start_part_id,finish_part_id,degree_id,degree_name_sl,finish_status,gpa_points,end_agpa_points,total_reg_credits,total_pass_credits
75,133.111,20111.0,20111.0,20134.0,19.111,إدارة الأعمال,CHANGE_PLAN,2.68,2.68,0.0,0.0
76,133.111,20112.0,20111.0,20134.0,19.111,إدارة الأعمال,CHANGE_PLAN,2.72,2.70,17.0,17.0
77,133.111,20121.0,20111.0,20134.0,19.111,إدارة الأعمال,CHANGE_PLAN,0.00,2.70,35.0,35.0
78,133.111,20122.0,20111.0,20134.0,19.111,إدارة الأعمال,CHANGE_PLAN,0.00,2.70,35.0,35.0
79,133.111,20131.0,20111.0,20134.0,19.111,إدارة الأعمال,CHANGE_PLAN,2.67,2.69,35.0,35.0
...,...,...,...,...,...,...,...,...,...,...,...
1418,240.111,20183.0,20153.0,20192.0,11.111,المحاسبة و التدقيق,GRADUATED,1.78,2.22,219.0,129.0
1419,240.111,20191.0,20153.0,20192.0,11.111,المحاسبة و التدقيق,GRADUATED,2.26,2.28,227.0,134.0
1420,240.111,20192.0,20153.0,20192.0,11.111,المحاسبة و التدقيق,GRADUATED,2.49,2.46,245.0,150.0
2234,1437.111,20111.0,20111.0,NaN,19.111,إدارة الأعمال,CLOSE_FILE,1.35,1.35,0.0,0.0
